# SDN Load Balancing Analysis

This notebook provides analysis and visualization of the SDN load balancing results.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set style
try:
    plt.style.use('seaborn-v0_8')
except OSError:
    try:
        plt.style.use('seaborn')
    except OSError:
        plt.style.use('default')
sns.set_palette("husl")

# Set paths
data_dir = '../data'
results_dir = '../experiments'
models_dir = '../models'


## 1. Load Results


In [ ]:
# Load evaluation results
results_path = os.path.join(results_dir, 'results.csv')
if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    print("Evaluation Results:")
    print(results_df.to_string())
else:
    print("Results file not found. Run experiments/evaluate.py first.")


## 2. TFT Prediction Analysis


In [ ]:
# Load TFT predictions
forecast_path = os.path.join(data_dir, 'predictions', 'tft_forecast.csv')
if os.path.exists(forecast_path):
    forecast_df = pd.read_csv(forecast_path)
    
    # Plot predictions vs actual
    if 'actual' in forecast_df.columns and 'predicted' in forecast_df.columns:
        plt.figure(figsize=(12, 6))
        
        # Sample for visualization if too many points
        sample_size = min(1000, len(forecast_df))
        sample_df = forecast_df.sample(n=sample_size) if len(forecast_df) > sample_size else forecast_df
        
        plt.plot(sample_df['actual'].values, label='Actual', alpha=0.7)
        plt.plot(sample_df['predicted'].values, label='Predicted', alpha=0.7)
        plt.xlabel('Sample')
        plt.ylabel('Throughput (kbps)')
        plt.title('TFT Predictions vs Actual')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Calculate metrics
        actual = forecast_df['actual'].dropna()
        predicted = forecast_df['predicted'].dropna()
        
        if len(actual) > 0 and len(predicted) > 0:
            mae = np.mean(np.abs(actual - predicted))
            mape = np.mean(np.abs((actual - predicted) / (actual + 1e-8))) * 100
            
            print(f"MAE: {mae:.4f}")
            print(f"MAPE: {mape:.2f}%")
else:
    print("Forecast file not found. Train TFT model first.")


## 3. Traffic Data Analysis


In [ ]:
# Load processed data
processed_path = os.path.join(data_dir, 'processed', 'processed.csv')
if os.path.exists(processed_path):
    data_df = pd.read_csv(processed_path)
    
    # Parse time if available
    if 'time' in data_df.columns:
        data_df['time'] = pd.to_datetime(data_df['time'])
        data_df = data_df.sort_values('time')
    
    # Plot traffic over time
    if 'throughput_kbps' in data_df.columns:
        plt.figure(figsize=(14, 8))
        
        # Group by link if available
        if 'link_id' in data_df.columns:
            for link_id in data_df['link_id'].unique()[:5]:  # Show first 5 links
                link_data = data_df[data_df['link_id'] == link_id]
                if 'time' in link_data.columns:
                    plt.plot(link_data['time'], link_data['throughput_kbps'], 
                            label=f'Link {link_id}', alpha=0.7)
                else:
                    plt.plot(link_data['throughput_kbps'], label=f'Link {link_id}', alpha=0.7)
        else:
            plt.plot(data_df['throughput_kbps'], alpha=0.7)
        
        plt.xlabel('Time')
        plt.ylabel('Throughput (kbps)')
        plt.title('Traffic Throughput Over Time')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Statistics
        print("Traffic Statistics:")
        stat_cols = ['throughput_kbps', 'latency_ms', 'packet_loss']
        available_cols = [col for col in stat_cols if col in data_df.columns]
        if available_cols:
            print(data_df[available_cols].describe())
        else:
            print("Required columns not found in data.")
else:
    print("Processed data not found. Run utils/preprocessing.py first.")


## 4. Algorithm Comparison


In [ ]:
# Load and visualize comparison results
results_path = os.path.join(results_dir, 'results.csv')
if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    # Create comparison plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    algorithms = results_df['Algorithm'].values
    
    # Reward comparison
    axes[0, 0].bar(algorithms, results_df['Avg_Reward'], 
                   yerr=results_df.get('Std_Reward', 0), capsize=5)
    axes[0, 0].set_title('Average Reward')
    axes[0, 0].set_ylabel('Reward')
    axes[0, 0].tick_params(axis='x', rotation=45)
    axes[0, 0].grid(True, alpha=0.3)
    
    # Throughput comparison
    axes[0, 1].bar(algorithms, results_df['Avg_Throughput'], 
                   yerr=results_df.get('Std_Throughput', 0), capsize=5)
    axes[0, 1].set_title('Average Throughput')
    axes[0, 1].set_ylabel('Throughput (kbps)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(True, alpha=0.3)
    
    # Latency comparison
    axes[1, 0].bar(algorithms, results_df['Avg_Latency'], 
                   yerr=results_df.get('Std_Latency', 0), capsize=5)
    axes[1, 0].set_title('Average Latency')
    axes[1, 0].set_ylabel('Latency (ms)')
    axes[1, 0].tick_params(axis='x', rotation=45)
    axes[1, 0].grid(True, alpha=0.3)
    
    # Packet loss comparison
    axes[1, 1].bar(algorithms, results_df['Avg_Packet_Loss'], 
                   yerr=results_df.get('Std_Packet_Loss', 0), capsize=5)
    axes[1, 1].set_title('Average Packet Loss')
    axes[1, 1].set_ylabel('Packet Loss')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'detailed_comparison.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nDetailed Comparison:")
    print(results_df.to_string(index=False))
else:
    print("Results file not found. Run experiments/evaluate.py first.")


## 5. Summary and Conclusions


In [ ]:
print("=== Analysis Summary ===")
print("\n1. TFT Model:")
print("   - Provides traffic forecasting for proactive load balancing")
print("   - Enables DQN agent to make informed routing decisions")

print("\n2. DQN Agent:")
print("   - Learns optimal routing policies through reinforcement learning")
print("   - Adapts to network conditions dynamically")

print("\n3. Baseline Comparisons:")
print("   - Round-Robin: Simple but may not optimize for network conditions")
print("   - Weighted Round-Robin: Better than RR but still static")
print("   - DQN: Adaptive and learns from experience")

print("\n4. Key Metrics:")
print("   - Throughput: Higher is better")
print("   - Latency: Lower is better")
print("   - Packet Loss: Lower is better")
print("   - Reward: Combined metric balancing all factors")
